# W5｜三道综合练习

课堂默认讲 A 题；B、C 供提前完成者继续探索，自学时可任选一题。三题互相独立。先展示题目和输入，停下来与 Agent 合作：预测结果、实现，再用自己的输入核对。向下的 **参考解答** 已公开，教师可在学生尝试之后展示；代码默认可见。

选择 `week-05/.venv` 的 Python kernel。每题先运行自己的输入 cell；整份可 **Restart Kernel → Run All**。这里展示示例，不会替你完成 `challenges/work.py`，作业仍需单独运行 student 验收。

## A｜保留首次顺序，统计重复情况

**目标：** 同一份数据，生成三种结果。

- 输入：整数列表，允许零、负数、重复值和空列表。
- 输出：`unique` 为首次出现顺序的新列表；`counts` 为频数字典；`repeated` 为出现超过一次的值组成的新 set。
- 不改变输入；返回的容器独立。set 只比较元素，不比较显示顺序。

```python
[3, 1, 3, 0, -1, 1, 3]
# 应得到：
{"unique": [3, 1, 0, -1],
 "counts": {3: 3, 1: 2, 0: 1, -1: 1},
 "repeated": {1, 3}}
```

**边界：** 空列表得到空列表、空字典和空 set；去重不能丢失首次顺序。

In [1]:
values = [3, 1, 3, 0, -1, 1, 3]
print("输入 values：", values)

输入 values： [3, 1, 3, 0, -1, 1, 3]


### 参考解答

第一次遇到一个值时记录顺序；每次遇到都累计频数，最后找出重复值。

In [2]:
def summarize_values(values):
    unique = []
    counts = {}
    for value in values:
        if value not in counts:
            unique.append(value)
            counts[value] = 0
        counts[value] += 1
    repeated = {value for value, count in counts.items() if count > 1}
    return {"unique": unique, "counts": counts, "repeated": repeated}

In [3]:
result = summarize_values(values)
print("unique：", result["unique"])
print("counts：", result["counts"])
print("repeated：", result["repeated"])

assert result == {
    "unique": [3, 1, 0, -1],
    "counts": {3: 3, 1: 2, 0: 1, -1: 1},
    "repeated": {1, 3},
}
assert values == [3, 1, 3, 0, -1, 1, 3]
assert sum(result["counts"].values()) == len(values)
print("样例核对：PASS；原输入保持：", values)

unique： [3, 1, 0, -1]
counts： {3: 3, 1: 2, 0: 1, -1: 1}
repeated： {1, 3}
样例核对：PASS；原输入保持： [3, 1, 3, 0, -1, 1, 3]


### 边界预测

`[2, 1, 2, 3, 1]` 的 unique 是 `[2, 1, 3]` 还是排序后的 `[1, 2, 3]`？空输入又返回什么？

In [4]:
ordered_input = [2, 1, 2, 3, 1]
ordered = summarize_values(ordered_input)
empty = summarize_values([])
print("顺序样例：", ordered)
print("空输入：", empty)

assert ordered == {
    "unique": [2, 1, 3],
    "counts": {2: 2, 1: 2, 3: 1},
    "repeated": {1, 2},
}
assert ordered_input == [2, 1, 2, 3, 1]
assert empty == {"unique": [], "counts": {}, "repeated": set()}
print("边界核对：PASS")

顺序样例： {'unique': [2, 1, 3], 'counts': {2: 2, 1: 2, 3: 1}, 'repeated': {1, 2}}
空输入： {'unique': [], 'counts': {}, 'repeated': set()}
边界核对：PASS


## B｜应用一批修改，报告最终变化

**目标：** 保留原记录，区分“操作过”与“最终改变”。

- 输入：字符串键、整数值的字典；有序的 `(键, 新值)` 更新列表；待删除键的列表。
- 先依次更新，同一键最后一次生效；再删除，缺失键或重复删除均忽略。
- 输出：新字典 `record`；相比原记录，最终新增、删除或值改变的键组成的 set `changed`。三份输入保持，结果独立。

```python
original = {"a": 1, "b": 2, "c": 3}
updates = [("b", 20), ("d", 4), ("b", 22)]
removals = ["c", "missing"]
# 应得到：
{"record": {"a": 1, "b": 22, "d": 4}, "changed": {"b", "c", "d"}}
```

**边界：** 改回原值、新增后删除，都不算最终变化；三份输入都可以为空。

In [5]:
original = {"a": 1, "b": 2, "c": 3}
updates = [("b", 20), ("d", 4), ("b", 22)]
removals = ["c", "missing"]
print("原记录：", original)
print("更新清单：", updates)
print("删除清单：", removals)

原记录： {'a': 1, 'b': 2, 'c': 3}
更新清单： [('b', 20), ('d', 4), ('b', 22)]
删除清单： ['c', 'missing']


### 参考解答

先在新字典上完成操作，再比较最终记录；不能用“处理过哪些键”代替比较。

In [6]:
def apply_changes(original, updates, removals):
    record = original.copy()
    for key, value in updates:
        record[key] = value
    for key in removals:
        record.pop(key, None)  # 缺失时忽略。

    changed = set()
    for key in original:
        if key not in record or original[key] != record[key]:
            changed.add(key)
    for key in record:
        if key not in original:
            changed.add(key)
    return {"record": record, "changed": changed}

In [7]:
original = {"a": 1, "b": 2, "c": 3}
updates = [("b", 20), ("d", 4), ("b", 22)]
removals = ["c", "missing"]

result = apply_changes(original, updates, removals)
print("最终记录：", result["record"])
print("最终变化的键：", result["changed"])

assert result == {
    "record": {"a": 1, "b": 22, "d": 4},
    "changed": {"b", "c", "d"},
}
assert original == {"a": 1, "b": 2, "c": 3}
assert updates == [("b", 20), ("d", 4), ("b", 22)]
assert removals == ["c", "missing"]
print("样例核对：PASS；三份输入保持原样")

最终记录： {'a': 1, 'b': 22, 'd': 4}
最终变化的键： {'b', 'c', 'd'}
样例核对：PASS；三份输入保持原样


### 边界预测

a 从 1 改成 9 又改回 1，new 新增后又删除：最终记录是什么，changed 应包含谁？

In [8]:
original = {"a": 1}
updates = [("a", 9), ("a", 1), ("new", 5)]
removals = ["new", "missing", "new"]
result = apply_changes(original, updates, removals)
print("最终记录：", result["record"])
print("最终变化的键：", result["changed"])

assert result == {"record": {"a": 1}, "changed": set()}
assert original == {"a": 1}
assert updates == [("a", 9), ("a", 1), ("new", 5)]
assert removals == ["new", "missing", "new"]
assert apply_changes({}, [], []) == {"record": {}, "changed": set()}
print("边界核对：PASS；操作过，不等于最终改变")

最终记录： {'a': 1}
最终变化的键： set()
边界核对：PASS；操作过，不等于最终改变


## C｜对齐两次测量，找出超出阈值的变化

**目标：** 保持编号对应，区分有正负的变化量与变化幅度。

- 输入：不重复的字符串编号；两组等长的一维整数数组，与编号逐项对应；非负整数 threshold。
- 测量值在 -1,000,000～1,000,000；三组数据可同时为空。
- `delta = current - baseline`；仅 `abs(delta) > threshold` 异常，等于阈值正常。
- 输出：新整数数组 `delta`；保持原相对顺序的异常编号 `alerts`；正常编号及其 **current 值** 的字典 `normal`。输入保持，结果独立。

```python
ids = ["a", "b", "c", "d"]
baseline = [10, 20, 30, 40]  # 两组测量均使用 np.array。
current  = [11, 18, 30, 45]; threshold = 2
# 应得到：delta = [1, -2, 0, 5]
# alerts = ["d"]; normal = {"a": 11, "b": 18, "c": 30}
```

**边界：** 大幅下降也异常；空输入返回空整数数组、空列表和空字典。

In [9]:
import numpy as np

ids = ["a", "b", "c", "d"]
baseline = np.array([10, 20, 30, 40])
current = np.array([11, 18, 30, 45])
threshold = 2
print("编号：", ids)
print("原测量：", baseline)
print("新测量：", current)
print("阈值：", threshold)

编号： ['a', 'b', 'c', 'd']
原测量： [10 20 30 40]
新测量： [11 18 30 45]
阈值： 2


### 参考解答

先转成 Python int 做差，再收集为 int64 数组，避免较窄整数类型溢出；按编号逐项判断幅度。

In [10]:
def compare_measurements(ids, baseline, current, threshold):
    delta = np.array(
        [int(after) - int(before) for before, after in zip(baseline, current)],
        dtype=np.int64,
    )
    alerts = []
    normal = {}
    for name, change, value in zip(ids, delta, current):
        if abs(int(change)) > threshold:
            alerts.append(name)
        else:
            normal[name] = int(value)
    return {"delta": delta, "alerts": alerts, "normal": normal}

In [11]:
ids = ["a", "b", "c", "d"]
baseline = np.array([10, 20, 30, 40])
current = np.array([11, 18, 30, 45])
threshold = 2

result = compare_measurements(ids, baseline, current, threshold)
print("delta：", result["delta"])
print("alerts：", result["alerts"])
print("normal：", result["normal"])

assert np.array_equal(result["delta"], np.array([1, -2, 0, 5]))
assert result["delta"].dtype.kind in "iu"
assert result["alerts"] == ["d"]
assert result["normal"] == {"a": 11, "b": 18, "c": 30}
assert ids == ["a", "b", "c", "d"]
assert np.array_equal(baseline, np.array([10, 20, 30, 40]))
assert np.array_equal(current, np.array([11, 18, 30, 45]))
print("样例核对：PASS；-2 恰等于阈值，仍属正常；输入保持")

delta： [ 1 -2  0  5]
alerts： ['d']
normal： {'a': 11, 'b': 18, 'c': 30}
样例核对：PASS；-2 恰等于阈值，仍属正常；输入保持


### 边界预测

int8 输入从 120 降到 -120，变化量会是 -240 吗？阈值为 3 时，变化 -3 是否异常？先预测下格的三个结果。

In [12]:
ids = ["down", "equal", "up"]
baseline = np.array([120, 100, -120], dtype=np.int8)
current = np.array([-120, 97, 120], dtype=np.int8)
threshold = 3
result = compare_measurements(ids, baseline, current, threshold)
print("输入 dtype：", baseline.dtype, "；输出 dtype：", result["delta"].dtype)
print("delta：", result["delta"])
print("alerts：", result["alerts"])
print("normal：", result["normal"])

assert np.array_equal(result["delta"], np.array([-240, -3, 240]))
assert result["alerts"] == ["down", "up"]
assert result["normal"] == {"equal": 97}
assert np.array_equal(baseline, np.array([120, 100, -120], dtype=np.int8))
assert np.array_equal(current, np.array([-120, 97, 120], dtype=np.int8))
empty = compare_measurements(
    [], np.array([], dtype=int), np.array([], dtype=int), 0
)
assert empty["delta"].shape == (0,) and empty["delta"].dtype.kind in "iu"
assert empty["alerts"] == [] and empty["normal"] == {}
print("空输入：", empty)
print("边界核对：PASS；负向异常被发现，恰等阈值正常，空结果保持整数类型")

输入 dtype： int8 ；输出 dtype： int64
delta： [-240   -3  240]
alerts： ['down', 'up']
normal： {'equal': 97}
空输入： {'delta': array([], dtype=int64), 'alerts': [], 'normal': {}}
边界核对：PASS；负向异常被发现，恰等阈值正常，空结果保持整数类型


## 回到自己的实现

Notebook 示例通过，只说明上面的参考代码通过这些例子。请在 `challenges/work.py` 实现选中的题目，再运行对应验收；以下命令均从 `week-05/` 的 Terminal 执行：

```bash
# A：顺序去重与频数统计。
uv run --locked pytest -q challenges -k summarize --challenge-impl=student
# B：应用修改并报告最终变化。
uv run --locked pytest -q challenges -k apply_changes --challenge-impl=student
# C：对齐两次测量并判断异常。
uv run --locked pytest -q challenges -k compare_measurements --challenge-impl=student
```

任选一题即可。再给一份自己的输入、完整预期和一条边界解释；不要把 Notebook 的 Run All 当成作业验收。